# Detect incorrectly answered non-RAG questions

Choose a prediction CSV below. This notebook compares its `answer` values with the dataset's `correct_answer` values and lists every incorrect `question_id` in dataset order. Partial result files are supported.

In [7]:
from __future__ import annotations

import csv
import json
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"

# Change this filename to inspect another GPT-5.6 Terra non-RAG result file.
ANSWERS_PATH = PROJECT_ROOT / "non_rag" / "gpt-5.6-terra" / "answers_patient_education.csv"

print(f"Dataset: {DATASET_PATH}")
print(f"Answers: {ANSWERS_PATH}")

Dataset: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\data\cancermyth_screening_dataset.json
Answers: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\non_rag\gpt-5.6-terra\answers_patient_education.csv


In [8]:
def parse_boolean(value: str, *, question_id: str) -> bool:
    normalized = value.strip().lower()
    if normalized == "true":
        return True
    if normalized == "false":
        return False
    raise ValueError(f"Answer for question_id={question_id} must be true or false; got {value!r}.")


with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    dataset = json.load(dataset_file)

if not isinstance(dataset, list) or not dataset:
    raise ValueError("The dataset must be a non-empty JSON array.")

dataset_by_id: dict[str, dict] = {}
for record in dataset:
    question_id = str(record.get("id"))
    if question_id in dataset_by_id:
        raise ValueError(f"Duplicate question ID in dataset: {question_id}")
    if not isinstance(record.get("correct_answer"), bool):
        raise ValueError(f"correct_answer for question_id={question_id} must be Boolean.")
    dataset_by_id[question_id] = record

if not ANSWERS_PATH.is_file():
    raise FileNotFoundError(f"Answer file does not exist: {ANSWERS_PATH}")

predictions: dict[str, bool] = {}
with ANSWERS_PATH.open(newline="", encoding="utf-8") as answers_file:
    reader = csv.DictReader(answers_file)
    if reader.fieldnames != ["question_id", "answer"]:
        raise ValueError("The answer CSV must contain exactly: question_id, answer")
    for row in reader:
        question_id = row["question_id"].strip()
        if question_id not in dataset_by_id:
            raise ValueError(f"Unknown question_id in answer CSV: {question_id}")
        if question_id in predictions:
            raise ValueError(f"Duplicate question_id in answer CSV: {question_id}")
        predictions[question_id] = parse_boolean(row["answer"], question_id=question_id)

print(f"Loaded {len(predictions):,} predictions from {ANSWERS_PATH.name}.")

Loaded 735 predictions from answers_patient_education.csv.


In [9]:
wrong_question_ids = [
    record["id"]
    for record in dataset
    if str(record["id"]) in predictions
    and predictions[str(record["id"])] != record["correct_answer"]
]

print(f"Wrong answers: {len(wrong_question_ids):,} / {len(predictions):,}")
print("Wrong question IDs:")
wrong_question_ids

Wrong answers: 262 / 735
Wrong question IDs:


[4,
 6,
 7,
 14,
 16,
 19,
 24,
 26,
 27,
 33,
 34,
 36,
 42,
 49,
 51,
 56,
 60,
 67,
 68,
 71,
 72,
 73,
 77,
 78,
 79,
 80,
 83,
 85,
 88,
 89,
 90,
 91,
 92,
 93,
 95,
 97,
 98,
 101,
 102,
 105,
 106,
 110,
 111,
 113,
 116,
 117,
 119,
 121,
 123,
 124,
 127,
 129,
 130,
 131,
 135,
 137,
 142,
 143,
 144,
 146,
 148,
 150,
 151,
 152,
 153,
 154,
 155,
 158,
 161,
 162,
 164,
 165,
 166,
 167,
 168,
 174,
 175,
 180,
 185,
 186,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 201,
 204,
 205,
 206,
 209,
 210,
 216,
 221,
 222,
 225,
 226,
 227,
 228,
 230,
 232,
 236,
 238,
 241,
 247,
 249,
 251,
 252,
 262,
 263,
 264,
 265,
 268,
 269,
 272,
 274,
 280,
 281,
 283,
 285,
 287,
 290,
 292,
 298,
 299,
 300,
 301,
 303,
 304,
 306,
 307,
 315,
 316,
 317,
 321,
 327,
 328,
 329,
 335,
 340,
 346,
 362,
 367,
 369,
 371,
 373,
 379,
 382,
 388,
 390,
 395,
 399,
 402,
 404,
 412,
 415,
 416,
 417,
 419,
 423,
 425,
 441,
 444,
 445,
 446,
 448,
 450,
 451,
 452,
 453,
 455,
 4